### Live Incident Triage & Local AI Analyser

This core script acts as a real-time investigator console. It monitors your saved text logs for updates, restructures complex server data data-points on the fly, and manages the network pipe to your local/remote AI server for instant threat parsing.

* **Smart Data Flattening:** It unzips complex, nested data categories and lays them out into flat vertical lines so the AI can read 100% of the forensic parameters without hitting layout clipping limits.
* **Array Ambiguity Safeguard:** It standardises incoming metrics into clean text values before evaluation, eliminating data engine crashes when multi-element lists land in the stream.
* **Analyst Action Prompt:** The execution loop locks onto an active alert and goes on standby indefinitely until you type a manual triage note and press Enter, saving the results straight to your history database.



In [ ]:
import os
import sys
import time
import json
import requests
import pandas as pd
from datetime import datetime

# ==============================================================================
# PARAMETERS & EXTERNAL ROUTING SCHEMAS
# ==============================================================================
AI_IP = "192.x.x.x"  # Replace with your verified remote AI server IP
FILE_PREFIX = "wzuh_siem_logs"
MODEL_NAME = "llama3.2:3b"     # Verified target registry string
TRIAGE_HISTORY_FILE = "wzuh_siem_triage_history.csv"

# Simple state variable to track progress by checking raw text line counts
watcher_state = {
    'file_index': 1,
    'last_processed_line': 0,
    'active': True
}

# ==============================================================================
# REMOTE OLLAMA INFERENCE HANDSHAKE ENGINE
# ==============================================================================
def query_remote_ollama(model_name, alert_row_dict):
    """Packages alert records directly into plain text layout for network travel."""
    ollama_url = f"http://{AI_IP}:11434/api/generate"
    
    persona_system_prompt = (
        "You are a Senior Tier-3 SOC Incident Analyst. Review the security log provided. "
        "Provide a sharp, forensic assessment without any pleasantries. Use this layout:\n"
        "ASSESSMENT: [Malicious / Suspicious / False Positive]\n"
        "CONFIDENCE SCORE: [0% to 100%]\n"
        "TECHNICAL REASON: [One clear sentence breaking down the payload signature]"
    )
    
    flat_df = pd.json_normalize(alert_row_dict)
    # Fixed: Added explicit row bracket index to resolve the hidden AttributeError crash
    flat_dict = flat_df.iloc[0].to_dict()
    
    # Fixed: Replaced pd.notna with safe native string type checking to resolve array ambiguity crash
    alert_data_text = "\n".join([f"{key}: {value}" for key, value in flat_dict.items() if value is not None and str(value).strip() != "" and str(value) != "nan"])
    
    full_prompt = f"{persona_system_prompt}\n\n[SECURITY LOG DATA]\n{alert_data_text}"
    
    payload = {
        "model": model_name,
        "prompt": full_prompt,
        "stream": False,
        "options": {
            "num_predict": 100
        }
    }
    
    try:
        response = requests.post(ollama_url, json=payload, timeout=90)
        if response.status_code == 200:
            return response.json().get("response", "No analysis returned from model."), alert_data_text
        return f"Error: Status code {response.status_code}", alert_data_text
    except Exception as network_error:
        return f"Network Error: Unable to reach AI box. Details: {str(network_error)}", alert_data_text

# ==============================================================================
# RESILIENT JSON STREAM FOREGROUND WATCHER
# ==============================================================================
print(f"[*] JSON Stream Watcher Active. Monitoring {FILE_PREFIX}_1.json...")
print("[-] Processing data rows from line 0...")
print("-----------------------------------------------------------------")

try:
    while watcher_state['active']:
        current_filename = f"{FILE_PREFIX}_{watcher_state['file_index']}.json"
        next_filename = f"{FILE_PREFIX}_{watcher_state['file_index'] + 1}.json"
        
        if os.path.exists(next_filename) and os.path.exists(current_filename):
            try:
                with open(current_filename, 'r', encoding='utf-8', errors='ignore') as f:
                    total_lines = sum(1 for _ in f)
                if watcher_state['last_processed_line'] >= total_lines:
                    watcher_state['file_index'] += 1
                    watcher_state['last_processed_line'] = 0
                    current_filename = next_filename
                    print(f"\n[*] Rolled over to next active JSON log file: {current_filename}")
            except Exception as rollover_err:
                print(f"[-] Rollover Error: {str(rollover_err)}")

        if os.path.exists(current_filename):
            try:
                with open(current_filename, 'r', encoding='utf-8', errors='ignore') as active_file:
                    raw_lines = active_file.readlines()
                
                total_file_lines = len(raw_lines)
                
                if total_file_lines > watcher_state['last_processed_line']:
                    new_text_lines = raw_lines[watcher_state['last_processed_line']:total_file_lines]
                    
                    for idx, text_line in enumerate(new_text_lines):
                        current_line_idx = watcher_state['last_processed_line'] + idx
                        
                        alert_dict = json.loads(text_line.strip())
                        ai_response, flat_printed_log = query_remote_ollama(MODEL_NAME, alert_dict)
                        
                        print(f"\n[+] NEW EVENT PROCESSED NATIVELY: {current_filename} (Text Line {current_line_idx})")
                        print(f"Timestamp: {alert_dict.get('timestamp', 'N/A')}")
                        print(f"Rule ID:   {alert_dict.get('rule', {}).get('id', 'N/A') if isinstance(alert_dict.get('rule'), dict) else alert_dict.get('rule.id', 'N/A')} (Level {alert_dict.get('rule', {}).get('level', 'N/A') if isinstance(alert_dict.get('rule'), dict) else alert_dict.get('rule.level', 'N/A')})")
                        print(f"Details:   {alert_dict.get('rule', {}).get('description', 'N/A') if isinstance(alert_dict.get('rule'), dict) else alert_dict.get('rule.description', 'N/A')}")
                        print("-----------------------------------------------------------------")
                        print(f"AI ANALYSIS:\n{ai_response}")
                        print("-----------------------------------------------------------------")
                        print(f"FORENSIC TELEMETRY:\n{flat_printed_log}")
                        print("-----------------------------------------------------------------")
                        
                        analyst_notes = input("Type analyst notes and press Enter to commit: ").strip()
                        
                        enrichment_payload = {
                            'triage_timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                            'allocated_ai_model': MODEL_NAME,
                            'analyst_manual_comments': analyst_notes if analyst_notes else "Reviewed - No explicit notes logged.",
                            'ai_analysis_snapshot': ai_response
                        }
                        
                        flat_alert_df = pd.json_normalize(alert_dict)
                        # Fixed: Added explicit row bracket index to resolve the history storage writer crash
                        flat_alert_dict = flat_alert_df.iloc[0].to_dict()
                        
                        complete_triage_row = {**flat_alert_dict, **enrichment_payload}
                        triage_df = pd.DataFrame([complete_triage_row])
                        
                        write_header = not os.path.exists(TRIAGE_HISTORY_FILE)
                        triage_df.to_csv(TRIAGE_HISTORY_FILE, mode='a', header=write_header, index=False)
                        print(f"[ SUCCESS ] log entry committed to {TRIAGE_HISTORY_FILE}")
                        print("-" * 65)
                    
                    watcher_state['last_processed_line'] = total_file_lines
            except Exception as loop_error:
                print(f"[-] Active Tracking Loop Error Encountered: {str(loop_error)}")
        
        time.sleep(5)

except KeyboardInterrupt:
    print("\n[-] Ingestion monitor stopped safely by analyst command.")



[*] JSON Stream Watcher Active. Monitoring wzuh_siem_logs_1.json...
[-] Processing data rows from line 0...
-----------------------------------------------------------------

[+] NEW EVENT PROCESSED NATIVELY: wzuh_siem_logs_1.json (Text Line 0)
Timestamp: 2026-09-18T10:34:45.113+0000
Rule ID:   5557 (Level 5)
Details:   unix_chkpwd: Password check failed.
-----------------------------------------------------------------
AI ANALYSIS:
ASSESSMENT: Suspicious
CONFIDENCE SCORE: 80%
TECHNICAL REASON: The login attempt by user 'test' with a password check failure on Unix-based system is likely an authorized action, but the lack of clear context and absence of a specific error message in the log data raises concerns.

[SECURITY LOG DATA]

timestamp: 2026-09-18T10:34:45.113+0000
id: 1789727685
-----------------------------------------------------------------
FLATTENED FORENSIC TELEMETRY:
timestamp: 2026-09-18T10:34:45.113+0000
id: 1789727685.390580
full_log: 2026-09-18T11:34:44.244739+01:00 hos

Type analyst notes and press Enter to commit:  Analyst Testing


[ SUCCESS ] log entry committed to wzuh_siem_triage_history.csv
-----------------------------------------------------------------

[+] NEW EVENT PROCESSED NATIVELY: wzuh_siem_logs_1.json (Text Line 1)
Timestamp: 2026-09-18T10:34:45.221+0000
Rule ID:   5503 (Level 5)
Details:   PAM: User login failed.
-----------------------------------------------------------------
AI ANALYSIS:
ASSESSMENT: Suspicious
CONFIDENCE SCORE: 60%
TECHNICAL REASON: The log entry indicates a failed authentication attempt via SSH, which is unusual since the user (test) does not typically require an authentication failure to access their own account.

[SECURITY LOG DATA]
timestamp: 2026-09-18T10:34:45.221+0000
id: 1789727685.391022
full_log: 2026
-----------------------------------------------------------------
FLATTENED FORENSIC TELEMETRY:
timestamp: 2026-09-18T10:34:45.221+0000
id: 1789727685.391022
full_log: 2026-09-18T11:34:44.245020+01:00 host sshd-session[221239]: pam_unix(sshd:auth): authentication failure;

Type analyst notes and press Enter to commit:  Analyst Testing


[ SUCCESS ] log entry committed to wzuh_siem_triage_history.csv
-----------------------------------------------------------------

[+] NEW EVENT PROCESSED NATIVELY: wzuh_siem_logs_1.json (Text Line 2)
Timestamp: 2026-09-18T10:34:47.125+0000
Rule ID:   5760 (Level 5)
Details:   sshd: authentication failed.
-----------------------------------------------------------------
AI ANALYSIS:
ASSESSMENT: Suspicious
CONFIDENCE SCORE: 60%
TECHNICAL REASON: The payload signature indicates an invalid or brute-force SSH attempt, as evidenced by the "Failed password for test from 192.x.x.x port 54754 ssh2" message.

[SECURITY LOG DATA]
timestamp: 2026-09-18T10:34:47.125+0000
id: 1789727687.391580

-----------------------------------------------------------------
FLATTENED FORENSIC TELEMETRY:
timestamp: 2026-09-18T10:34:47.125+0000
id: 1789727687.391580
full_log: 2026-09-18T11:34:46.515369+01:00 host sshd-session[221239]: Failed password for test from 192.x.x.x port 54754 ssh2
location: /var/log/auth.l

Type analyst notes and press Enter to commit:  


[ SUCCESS ] log entry committed to wzuh_siem_triage_history.csv
-----------------------------------------------------------------

[+] NEW EVENT PROCESSED NATIVELY: wzuh_siem_logs_1.json (Text Line 3)
Timestamp: 2026-09-18T10:35:21.543+0000
Rule ID:   533 (Level 7)
Details:   Listened ports status (netstat) changed (new port opened or closed).
-----------------------------------------------------------------
AI ANALYSIS:
ASSESSMENT: Suspicious
CONFIDENCE SCORE: 20%
TECHNICAL REASON: The presence of a new, unexpected port (udp6 :::1716) associated with the kdeconnectd process is inconsistent with normal network behavior.

[SECURITY LOG DATA]

timestamp: 2026-09-18T10:35:21.543+0000
id: 1789727721.393031
previous_output: Previous output:
ossec: output
-----------------------------------------------------------------
FLATTENED FORENSIC TELEMETRY:
timestamp: 2026-09-18T10:35:21.543+0000
id: 1789727721.393031
previous_output: Previous output:
ossec: output: 'netstat listening ports':
tcp 12